In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [3]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [4]:
os.chdir("/home/lishaogang/NEU_2026")

In [5]:
adata = sc.read_h5ad('/home/lishaogang/data/final_adata_0123.h5ad')

In [ ]:
print(adata.obs['Disease'].unique().tolist())


In [ ]:
#adata中删除细胞数小于300的样本，除了Disease为CAR-T-DIPG的samples之外
# 1. 计算每个样本（orig.ident）的细胞总数
sample_sizes = adata.obs.groupby('orig.ident', observed=False)['orig.ident'].transform('count')

# 2. 定义筛选逻辑：
# 保留条件：(细胞数 >= 300) OR (Disease 等于 'CAR-T-DIPG')
mask = (sample_sizes >= 300) | (adata.obs['Disease'] == 'CAR-T-DIPG')

# 3. 执行过滤
adata = adata[mask].copy()

# 4. 打印筛选结果进行核对
print(f"筛选后剩余细胞数: {adata.n_obs}")
print("\n各样本剩余细胞统计:")
print(adata.obs['orig.ident'].value_counts())
print("\n各疾病组剩余样本统计:")
print(adata.obs.groupby('Disease', observed=False)['orig.ident'].nunique())

In [7]:
adata.obs['orig.ident'].unique().shape[0]

In [7]:
#adata.write_h5ad("adata_0126_temp.h5ad")

In [4]:
adata = sc.read_h5ad("adata_0126_temp.h5ad")

In [22]:
sc.pl.umap(adata, color='cell_type'
           #,legend_loc='on data'
           )

In [ ]:
adata.X = adata.layers['cp10k_counts'].copy()

In [24]:
# 前提：adata.X 已经是 log-normalized
sc.pp.log1p(adata) 

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='dataset',
    subset=False
)

adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, zero_center=False)
# 不 scale
sc.tl.pca(
    adata_hvg,
    n_comps=60,
    svd_solver='randomized',
    random_state=seed
)

sc.external.pp.harmony_integrate(
    adata_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.obsm['X_pca_harmony'] = adata_hvg.obsm['X_pca_harmony']


In [25]:
# 1. 针对高变基因重新运行 PCA
# 按照新版 Scanpy 的规范：使用 mask_var="highly_variable"
sc.tl.pca(adata, n_comps=50, mask_var="highly_variable")

# 2. 画碎石图 (保持不变)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [26]:


# --- 2d. 重新计算邻接图、UMAP 和聚类 ---
print("步骤 2d: 重新计算邻接图、UMAP 和 Leiden 聚类...")
# 使用新的 Harmony 结果
sc.pp.neighbors(adata,n_neighbors=30, n_pcs=30, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata, random_state=seed)

# 使用新的 key_added，以避免覆盖旧的聚类结果
sc.tl.leiden(adata, resolution=2.0, key_added='leiden', random_state=seed)

# 可视化新的精细聚类结果
print("可视化新的精细聚类结果...")
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Refined Cell Clusters (After Filtering)')


In [32]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Refined Cell Clusters (After Filtering)')


In [28]:
import matplotlib.pyplot as plt
import scanpy as sc

# 确保adata已加载
# adata = sc.read_h5ad('your_data.h5ad')

# 绘制多个color维度的UMAP（自动创建多面板）
# 1. 设置整体画布尺寸（ncols=2 让两个图并排，figsize控制整体大小）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  # 1行2列，总宽度16，高度8（保证每个子图8x8方形）

# 2. 分别绘制每个color维度，指定对应的ax
# 第一个面板：leiden
sc.pl.umap(adata, 
           color='leiden',
           legend_loc='on data', 
           title='Refined Cell Clusters',
           ax=axes[0],          # 指定第一个子图ax
           frameon=False, 
           show=False)

# 第二个面板：dataset
sc.pl.umap(adata, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],          # 指定第二个子图ax
           frameon=False, 
           show=False)

# 3. 强制每个子图都是正方形
for ax in axes:
    ax.set_box_aspect(1)

# 优化布局，避免重叠
plt.tight_layout()
plt.show()

In [13]:
# --- Define a core list of marker genes for major CSF populations ---
all_marker_genes_list = [
    # T cells
    "CD3D", "CD3E", "CD3G",

    # B cell
    "MS4A1", "CD79A", "CD19",

    # Plasma cell
    # Note: For Plasma cell, we list the positive markers.
    # The down-regulation of B cell markers (MS4A1, CD19) is checked against the B cell markers.
    "IGHG1", "MZB1", "SDC1",

    # Myeloid cell (for Monocytes and conventional DCs)
    "LYZ", "S100A8", "S100A9", "VCAN",

    # Plasmacytoid Dendritic Cells (pDC)
    "LILRA4", "IL3RA", "CLEC4C",

    # B cell
    "NKG7", "FCGR3A", "FGFBP2", "CX3CR1",'SLC4A10',
    
    # NKT cells
    "KLRB1",'TRBC1','TRBC2','TRAV10','ZBTB16',
    
    #T cell
    "CD8A", "CD4",
    
    # Gamma-Delta T cell
    "TRGV9", "TRDV2",

    # Platelet Markers (Potential Contamination)
    "PPBP", "PF4", "GP9", "ITGA2B",

    # Erythrocyte Markers (Potential Contamination)
    "HBB", "HBA1", "HBA2",
]
print(f"定义的 Marker 基因总数: {len(all_marker_genes_list)}")

In [33]:
adata.raw = adata.copy()  # 保存原始数据

In [29]:
# --- Check for marker gene availability in the data ---
# Use .raw.var_names as it contains all genes before HVG filtering.
available_genes_in_raw = adata.raw.var_names.tolist()
genes_to_plot = [gene for gene in all_marker_genes_list if gene in available_genes_in_raw]

skipped_genes = set(all_marker_genes_list) - set(genes_to_plot)
if skipped_genes:
    print(f"Warning: The following genes were not found in the data and will be skipped:\n{list(skipped_genes)}")

In [30]:
import importlib
import anno_self
importlib.reload(anno_self)

In [31]:
# 1. 设置全局图像大小为 8x8
sc.set_figure_params(figsize=(8, 8), dpi=100)

In [34]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden",
    standard_scale='var',
    cmap='Reds',
    use_raw=True
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


# --- 3b. 执行自动化注释 ---
# 假设您的 anno_self.annotate_cluster 函数仍然适用
# 如果您有更精细的髓系注释函数（如 annotate_CSF_myeloid），也可以在这里使用
print("步骤 3b: 执行自动化注释...")
annotations_list = final_summary_df.apply(anno_self.annotate_cluster, axis=1) # 或者用你更具体的函数


# Convert the annotation result (a Series) into a DataFrame and set column names
# reset_index() moves the original index (cluster IDs) into a column named 'index'
# rename() changes the column names to 'Cluster' (for cluster IDs) and 'Cell Type' (for annotations)
annotations_df = annotations_list.reset_index().rename(columns={'index': 'Cluster', 0: 'Cell Type'})


# Generate an explanation column for the annotation DataFrame
# Apply the generate_explanation function from anno_self row-wise to annotations_df
# This function typically creates a text explanation for how each cluster's cell type was determined

annotations_df['Explanation'] = annotations_df.apply(
    lambda row: anno_self.generate_explanation(row, final_summary_df),  # 额外传入 final_summary_df
    axis=1
)
# Print the annotated DataFrame using the custom pretty_print_df function from anno_self
# This function usually formats the output for better readability (e.g., adjusting column widths)
anno_self.pretty_print_df(annotations_df)




# --- 3c. 将注释映射回 AnnData 对象 ---
print("步骤 3c: 将注释映射回 AnnData 对象...")
annotation_dict = annotations_df.set_index('Cluster')['Cell Type'].to_dict()

# 创建一个新的 obs 列来存储精细化的注释
adata.obs['cell_type'] = adata.obs['leiden'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用leiden作为color，指定到axes[0]
sc.pl.umap(adata, 
           color='leiden',
           legend_loc='on data', 
           title='leiden Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [36]:
#保存adata
adata.write_h5ad("adata_0127.adata")

In [37]:

# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用leiden作为color，指定到axes[0]
sc.pl.umap(adata, 
           color='leiden',
           legend_loc='on data', 
           title='leiden Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata, 
           color='cell_subtype',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()

In [42]:
adata_NK = adata[adata.obs['cell_type'] == 'NK cell lineage'].copy()
print(f"Successfully extracted {adata_NK.n_obs} NK-cells for sub-clustering.")
adata_NK.write_h5ad("/home/lishaogang/NEU2026-Jan/adata_NK_raw.h5ad")

In [ ]:
# --- Subset T-cells for re-analysis ---
adata_T = adata[adata.obs['cell_type'] == 'T cell lineage'].copy()
print(f"Successfully extracted {adata_T.n_obs} T-cells for sub-clustering.")
adata_T.write_h5ad("/home/lishaogang/NEU2026-Jan/adata_T_raw.h5ad")

adata_B = adata[adata.obs['cell_type'] == 'B cell lineage'].copy()
print(f"Successfully extracted {adata_B.n_obs} B-cells for sub-clustering.")
adata_B.write_h5ad("/home/lishaogang/NEU2026-Jan/adata_B_raw.h5ad")

adata_Myeloid = adata[adata.obs['cell_type'] == 'Myeloid cell lineage'].copy()
print(f"Successfully extracted {adata_Myeloid.n_obs} Myeloid-cells for sub-clustering.")
adata_Myeloid.write_h5ad("/home/lishaogang/NEU2026-Jan/adata_Myeloid_raw.h5ad")

